# Explore here

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# paso 1 (planteamiento del problema y recopilacion de datos)
# lo que queremos predecir es si un estudiante va a abandonar el curso o no (dropout)
# es un problema de clasificacion binaria: dropout = 1 (abandona), dropout = 0 (no abandona)

REPO_ROOT = Path("..").resolve()
DATA_PATH = REPO_ROOT / "online_learning_engagement_dataset.csv"
DB_PATH   = REPO_ROOT / "online_learning.db"

In [ ]:
# paso 2 (obtener los datos desde la API de kaggle)
# si el archivo ya existe no lo descarga otra vez

if not DATA_PATH.exists():
    import subprocess, zipfile

    KAGGLE_DATASET = "rabieelkharoua/students-performance-dataset"

    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "-p", str(REPO_ROOT), "--unzip"],
        check=True
    )
    print(f"Dataset descargado en: {DATA_PATH}")
else:
    print(f"Dataset ya disponible en: {DATA_PATH}")

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

In [ ]:
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())

# no hay valores nulos

In [ ]:
# CONCLUSION paso 2: los datos se obtuvieron correctamente desde la API de kaggle. El dataset tiene mas de 60,000 filas
# y 20+ columnas, cumple con los requisitos. No hay valores nulos asi que no hay que limpiar nada por ahora

In [ ]:
# paso 3 (almacenamiento en base de datos SQLite)
# dividimos el dataset en 2 tablas para poder practicar JOINs
# students: datos demograficos y de acceso
# performance: metricas academicas y de engagement

from sqlalchemy import create_engine, text

engine = create_engine(f"sqlite:///{DB_PATH}")

students = df[["student_id", "age", "gender", "country",
               "device_type", "internet_speed_mbps",
               "study_hours_weekly", "login_frequency_weekly"]].copy()

performance = df[["student_id", "avg_session_duration_min", "video_watch_time_min",
                  "assignments_submitted", "forum_posts", "quiz_attempts",
                  "avg_quiz_score", "attendance_rate", "engagement_score",
                  "final_grade", "dropout"]].copy()

students.to_sql("students", engine, if_exists="replace", index=False)
performance.to_sql("performance", engine, if_exists="replace", index=False)

print(f"Base de datos creada en: {DB_PATH}")
print(f"  Tabla students:    {len(students):,} registros")
print(f"  Tabla performance: {len(performance):,} registros")

In [ ]:
# consultas SQL - SELECT

# distribucion de estudiantes por pais (top 10)
query_paises = """
SELECT country,
       COUNT(*) AS total_students,
       ROUND(AVG(study_hours_weekly), 2) AS avg_study_hours
FROM students
GROUP BY country
ORDER BY total_students DESC
LIMIT 10
"""
df_paises = pd.read_sql(query_paises, engine)
print("Top 10 paises por numero de estudiantes:")
df_paises

In [ ]:
# tasa de abandono por dispositivo
query_dropout = """
SELECT device_type,
       COUNT(*) AS total,
       SUM(p.dropout) AS dropouts,
       ROUND(100.0 * SUM(p.dropout) / COUNT(*), 2) AS dropout_rate_pct
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY device_type
ORDER BY dropout_rate_pct DESC
"""
df_dropout = pd.read_sql(query_dropout, engine)
print("Tasa de abandono por tipo de dispositivo:")
df_dropout

In [ ]:
# consultas SQL - JOIN

# estudiantes con alto riesgo de abandono (engagement bajo y asistencia < 50%)
query_riesgo = """
SELECT s.student_id, s.age, s.gender, s.country, s.study_hours_weekly,
       p.engagement_score, p.attendance_rate, p.avg_quiz_score,
       p.final_grade, p.dropout
FROM students s
INNER JOIN performance p ON s.student_id = p.student_id
WHERE p.attendance_rate < 0.5
  AND p.engagement_score < 5
ORDER BY p.engagement_score ASC
LIMIT 20
"""
df_riesgo = pd.read_sql(query_riesgo, engine)
print(f"Estudiantes de alto riesgo: {len(df_riesgo)} mostrados")
df_riesgo

In [ ]:
# rendimiento promedio por genero y velocidad de internet
query_internet = """
SELECT s.gender,
       CASE WHEN s.internet_speed_mbps >= 50 THEN 'Alta (>=50 Mbps)'
            ELSE 'Baja (<50 Mbps)' END AS internet_tier,
       COUNT(*) AS total_students,
       ROUND(AVG(p.final_grade), 2)      AS avg_final_grade,
       ROUND(AVG(p.avg_quiz_score), 2)   AS avg_quiz_score,
       ROUND(AVG(p.engagement_score), 2) AS avg_engagement
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY s.gender, internet_tier
ORDER BY s.gender, internet_tier
"""
df_internet = pd.read_sql(query_internet, engine)
print("Rendimiento promedio por genero y velocidad de internet:")
df_internet

In [ ]:
# consultas SQL - INSERT
# insertamos un estudiante ficticio para demostrar escritura en la BD

with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO students
            (student_id, age, gender, country, device_type,
             internet_speed_mbps, study_hours_weekly, login_frequency_weekly)
        VALUES (999999, 28, 'Male', 'Spain', 'Laptop', 100.0, 20.0, 7)
    """))

    conn.execute(text("""
        INSERT INTO performance
            (student_id, avg_session_duration_min, video_watch_time_min,
             assignments_submitted, forum_posts, quiz_attempts,
             avg_quiz_score, attendance_rate, engagement_score,
             final_grade, dropout)
        VALUES (999999, 60.0, 120.0, 10, 15, 8, 85.0, 0.95, 9.0, 90.0, 0)
    """))

    conn.commit()

verify = pd.read_sql("SELECT * FROM students WHERE student_id = 999999", engine)
print("Registro insertado:")
verify

In [ ]:
# reconstruimos el dataset completo con un JOIN de las dos tablas

df_full = pd.read_sql("""
    SELECT s.student_id, s.age, s.gender, s.country, s.device_type,
           s.internet_speed_mbps, s.study_hours_weekly, s.login_frequency_weekly,
           p.avg_session_duration_min, p.video_watch_time_min,
           p.assignments_submitted, p.forum_posts, p.quiz_attempts,
           p.avg_quiz_score, p.attendance_rate, p.engagement_score,
           p.final_grade, p.dropout
    FROM students s
    JOIN performance p ON s.student_id = p.student_id
    ORDER BY s.student_id
""", engine)

print(f"Total registros: {len(df_full):,}")
df_full

In [ ]:
# CONCLUSION paso 3: los datos se guardaron correctamente en SQLite con dos tablas relacionadas por student_id.
# las consultas SQL nos muestran que la distribucion de estudiantes por pais es bastante homogenea,
# la tasa de dropout es similar entre dispositivos, y los estudiantes con bajo engagement y baja asistencia
# tienen mayor probabilidad de abandonar. El INSERT demuestra que podemos escribir en la BD

In [ ]:
# paso 4 (analisis descriptivo)

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# quitamos el registro ficticio que insertamos antes
df_full = df_full[df_full["student_id"] != 999999].copy()

num_cols = ["age", "internet_speed_mbps", "study_hours_weekly",
            "login_frequency_weekly", "avg_session_duration_min",
            "video_watch_time_min", "assignments_submitted", "forum_posts",
            "quiz_attempts", "avg_quiz_score", "attendance_rate",
            "engagement_score", "final_grade"]

df_full[num_cols].describe()

In [ ]:
# variables categoricas: gender, country, device_type

fig, axis = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(ax=axis[0], data=df_full, x='gender')
sns.histplot(ax=axis[1], data=df_full, x='device_type')
sns.histplot(ax=axis[2], data=df_full, x='country')
axis[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# variables numericas: histograma + boxplot 

fig, axis = plt.subplots(len(num_cols), 2, figsize=(12, 4 * len(num_cols)),
                         gridspec_kw={"width_ratios": [3, 1]})

for i, col in enumerate(num_cols):
    sns.histplot(ax=axis[i, 0], data=df_full, x=col, kde=True)
    sns.boxplot(ax=axis[i, 1], data=df_full, y=col)

plt.tight_layout()
plt.show()

# las distribuciones se ven bastante uniformes, tiene sentido porque el dataset es sintetico

In [ ]:
# CONCLUSION paso 4: las estadisticas descriptivas muestran que todas las variables numericas tienen dispersion moderada
# y no hay valores extremos evidentes. Las distribuciones son uniformes lo cual es tipico de datasets sinteticos.
# Las variables categoricas estan balanceadas entre sus categorias. No se requieren transformaciones previas al EDA

In [ ]:
# paso 5 (EDA)
# eliminamos student_id porque es un identificador, no sirve como variable predictora

df_eda = df_full.drop(columns=["student_id"]).copy()

print(f"Dataset para EDA: {df_eda.shape[0]:,} filas x {df_eda.shape[1]} columnas")
print(f"\nDistribucion de dropout:")
print(df_eda['dropout'].value_counts())
print(f"\nProporcion de dropout: {df_eda['dropout'].mean()*100:.2f}%")

In [ ]:
# exploracion y limpieza

print("Valores nulos por columna:")
print(df_eda.isnull().sum())

duplicados = df_eda.duplicated().sum()
print(f"\nFilas duplicadas: {duplicados}")
if duplicados > 0:
    df_eda = df_eda.drop_duplicates().reset_index(drop=True)
    print(f"  Eliminadas. Nuevo shape: {df_eda.shape}")

print(f"\nTipos de datos:")
print(df_eda.dtypes)

In [ ]:
# deteccion de outliers con IQR

num_cols_eda = df_eda.select_dtypes(include=[np.number]).columns.tolist()
num_cols_eda = [c for c in num_cols_eda if c != "dropout"]

outlier_summary = []
for col in num_cols_eda:
    Q1 = df_eda[col].quantile(0.25)
    Q3 = df_eda[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_eda[col] < lower) | (df_eda[col] > upper)).sum()
    outlier_summary.append({
        "variable": col,
        "Q1": round(Q1, 2),
        "Q3": round(Q3, 2),
        "IQR": round(IQR, 2),
        "limite_inferior": round(lower, 2),
        "limite_superior": round(upper, 2),
        "n_outliers": n_outliers
    })

pd.DataFrame(outlier_summary)

In [ ]:
# boxplots para ver outliers visualmente

fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    df_eda.boxplot(column=col, ax=axes[i], vert=True)
    axes[i].set_title(col, fontsize=10)

for j in range(len(num_cols_eda), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# analisis univariante - variables numericas separadas por dropout

fig, axes = plt.subplots(4, 4, figsize=(22, 18))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    ax = axes[i]
    for label, color in [(0, "steelblue"), (1, "salmon")]:
        subset = df_eda[df_eda["dropout"] == label][col]
        ax.hist(subset, bins=35, alpha=0.5, color=color, label=f"dropout={label}", density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

for j in range(len(num_cols_eda), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# variables categoricas por dropout

cat_cols_eda = ["gender", "device_type"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(cat_cols_eda):
    sns.countplot(data=df_eda, x=col, hue="dropout", ax=axes[i])
    axes[i].set_title(f"{col} por dropout")

plt.tight_layout()
plt.show()

# country tiene muchos valores entonces mejor mostramos la tasa de dropout por pais
dropout_by_country = df_eda.groupby("country")["dropout"].agg(["mean", "count"]).sort_values("count", ascending=False).head(10)
dropout_by_country.columns = ["dropout_rate", "total_students"]
dropout_by_country["dropout_rate"] = (dropout_by_country["dropout_rate"] * 100).round(2)
print("\nTasa de dropout por pais (top 10):")
dropout_by_country

In [ ]:
# analisis multivariante
# numercicos - numericos

pairs = [
    ("engagement_score", "final_grade"),
    ("attendance_rate", "final_grade"),
    ("study_hours_weekly", "final_grade"),
    ("avg_quiz_score", "engagement_score"),
]

fig, axis = plt.subplots(len(pairs), 2, figsize=(12, 5 * len(pairs)))

for i, (x_var, y_var) in enumerate(pairs):
    sns.regplot(ax=axis[i, 0], data=df_eda, x=x_var, y=y_var, scatter_kws={"alpha": 0.1, "s": 5})
    sns.heatmap(df_eda[[x_var, y_var]].corr(), annot=True, fmt=".2f", ax=axis[i, 1], cbar=False)

plt.tight_layout()
plt.show()

In [ ]:
# categoricos - categoricos

sns.countplot(data=df_eda, x="device_type", hue="gender")
plt.show()

In [ ]:
# numerico vs categorico: heatmap general con factorize

df_corr = df_eda.copy()
df_corr["gender"] = pd.factorize(df_corr["gender"])[0]
df_corr["device_type"] = pd.factorize(df_corr["device_type"])[0]
df_corr["country"] = pd.factorize(df_corr["country"])[0]

corr_cols = ["gender", "country", "device_type"] + num_cols_eda + ["dropout"]
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)

plt.tight_layout()
plt.show()

# correlacion con dropout ordenada por valor absoluto
corr_target = corr_matrix["dropout"].drop("dropout").abs().sort_values(ascending=False)
print("\nCorrelacion de cada variable con dropout:")
corr_target

In [ ]:
# pairplot con las 5 variables mas correlacionadas con dropout

top5_vars = corr_target.head(5).index.tolist() + ["dropout"]
sns.pairplot(df_eda[top5_vars], hue="dropout", diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10})
plt.show()

In [ ]:
# CONCLUSION analisis: las correlaciones con dropout son bajas en general lo que indica que no hay una variable
# que por si sola explique el abandono. Esto tiene sentido porque la decision de abandonar depende de multiples factores.
# gender y device_type no parecen afectar el dropout, country tampoco muestra diferencias grandes entre paises

In [ ]:
# paso 5.5 (ingenieria de caracteristicas)
# convertimos las categoricas a numeros con factorize

df_fe = df_eda.copy()

df_fe["gender"] = pd.factorize(df_fe["gender"])[0]
df_fe["device_type"] = pd.factorize(df_fe["device_type"])[0]
df_fe["country"] = pd.factorize(df_fe["country"])[0]

df_fe.head()

In [ ]:
# escalamos las variables con MinMaxScaler para que esten entre 0 y 1

from sklearn.preprocessing import MinMaxScaler

features_to_scale = [c for c in df_fe.columns if c != "dropout"]

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_fe[features_to_scale]),
    index=df_fe.index, columns=features_to_scale
)
df_scaled["dropout"] = df_fe["dropout"]

df_scaled.head()

In [ ]:
# paso 6 (seleccion de caracteristicas)
# dividimos los datos ANTES de seleccionar features para evitar data leakage

from sklearn.feature_selection import chi2, SelectKBest
from sklearn.model_selection import train_test_split

X = df_scaled.drop("dropout", axis=1)
y = df_scaled["dropout"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# probamos distintos valores de k para ver cual da mejor accuracy

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

resultados = {}

for k in range(1, X_train.shape[1] + 1):
    selector = SelectKBest(score_func=chi2, k=k)
    X_train_k = selector.fit_transform(X_train, y_train)
    X_test_k = selector.transform(X_test)

    modelo = LogisticRegression(max_iter=1000, random_state=42)
    modelo.fit(X_train_k, y_train)
    y_pred = modelo.predict(X_test_k)
    resultados[k] = accuracy_score(y_test, y_pred)

plt.figure(figsize=(10, 5))
plt.plot(list(resultados.keys()), list(resultados.values()), marker="o")
plt.xlabel("Numero de features (k)")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

best_k = max(resultados, key=resultados.get)
print(f"\nMejor k: {best_k} (Accuracy: {resultados[best_k]:.4f})")

In [ ]:
# aplicamos SelectKBest con el mejor k

selector = SelectKBest(score_func=chi2, k=best_k)
X_train_sel = pd.DataFrame(
    selector.fit_transform(X_train, y_train),
    columns=X_train.columns[selector.get_support()]
)
X_test_sel = pd.DataFrame(
    selector.transform(X_test),
    columns=X_train.columns[selector.get_support()]
)

print(f"Features seleccionadas ({best_k}): {list(X_train_sel.columns)}")
print(f"Train: {X_train_sel.shape}")
print(f"Test:  {X_test_sel.shape}")

In [ ]:
# guardamos los datos limpios como csv

train_export = X_train_sel.copy()
train_export["dropout"] = y_train.values
test_export = X_test_sel.copy()
test_export["dropout"] = y_test.values

train_export.to_csv("../data/processed/train_limpio.csv", index=False)
test_export.to_csv("../data/processed/test_limpio.csv", index=False)

print(f"train_limpio.csv ({train_export.shape})")
print(f"test_limpio.csv ({test_export.shape})")

In [ ]:
# CONCLUSION paso 5-6: el dataset esta limpio (sin nulos ni duplicados). Despues de escalar con MinMaxScaler
# y aplicar SelectKBest con chi2 nos quedamos con las features mas relevantes para predecir dropout.
# los datos estan divididos 80/20 y exportados como CSV listos para el modelado

In [ ]:
# paso 7 (construccion del modelo)
# vamos a probar 3 modelos: regresion logistica, random forest y gradient boosting
# primero los entrenamos sin optimizar para tener una linea base

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)

baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest":       RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

baseline_results = []

for name, model in baseline_models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    baseline_results.append({
        "Modelo": name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob), 4),
    })

df_baseline = pd.DataFrame(baseline_results).set_index("Modelo")
df_baseline

In [ ]:
# optimizacion de logistic regression con GridSearchCV

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

param_grid_lr = {
    "C":       [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver":  ["liblinear", "saga"],
}

gs_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid_lr, scoring="f1", cv=5, n_jobs=-1
)
gs_lr.fit(X_train_sel, y_train)

print(f"Mejores hiperparametros: {gs_lr.best_params_}")
print(f"Mejor F1-Score (CV): {gs_lr.best_score_:.4f}")

best_lr = gs_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test_sel)
y_prob_lr = best_lr.predict_proba(X_test_sel)[:, 1]

print(classification_report(y_test, y_pred_lr))

In [ ]:
# optimizacion de random forest con RandomizedSearchCV

from scipy.stats import randint as sp_randint

param_dist_rf = {
    "n_estimators":      sp_randint(100, 500),
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": sp_randint(2, 20),
    "min_samples_leaf":  sp_randint(1, 10),
    "max_features":      ["sqrt", "log2", None],
    "class_weight":      [None, "balanced"],
}

rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_rf, n_iter=50, scoring="f1", cv=5, random_state=42, n_jobs=-1
)
rs_rf.fit(X_train_sel, y_train)

print(f"Mejores hiperparametros: {rs_rf.best_params_}")
print(f"Mejor F1-Score (CV): {rs_rf.best_score_:.4f}")

best_rf = rs_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_sel)
y_prob_rf = best_rf.predict_proba(X_test_sel)[:, 1]

print(classification_report(y_test, y_pred_rf))

In [ ]:
# optimizacion de gradient boosting con RandomizedSearchCV

from scipy.stats import uniform as sp_uniform

param_dist_gb = {
    "n_estimators":   sp_randint(100, 400),
    "learning_rate":  sp_uniform(0.01, 0.3),
    "max_depth":      sp_randint(3, 8),
    "subsample":      sp_uniform(0.6, 0.4),
    "min_samples_split": sp_randint(2, 20),
    "max_features":   ["sqrt", "log2", None],
}

rs_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist_gb, n_iter=50, scoring="f1", cv=5, random_state=42, n_jobs=-1
)
rs_gb.fit(X_train_sel, y_train)

print(f"Mejores hiperparametros: {rs_gb.best_params_}")
print(f"Mejor F1-Score (CV): {rs_gb.best_score_:.4f}")

best_gb = rs_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test_sel)
y_prob_gb = best_gb.predict_proba(X_test_sel)[:, 1]

print(classification_report(y_test, y_pred_gb))

In [ ]:
# comparacion final de los 3 modelos optimizados

results_opt = [
    ("Logistic Regression", y_pred_lr, y_prob_lr),
    ("Random Forest",       y_pred_rf, y_prob_rf),
    ("Gradient Boosting",   y_pred_gb, y_prob_gb),
]

comparison = []
for name, y_pred, y_prob in results_opt:
    comparison.append({
        "Modelo": name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob), 4),
    })

df_comparison = pd.DataFrame(comparison).set_index("Modelo")
df_comparison

In [ ]:
# grafico comparativo

fig, ax = plt.subplots(figsize=(10, 5))
df_comparison[["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]].T.plot(
    kind="bar", ax=ax, edgecolor="white")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# matrices de confusion

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred, _) in zip(axes, results_opt):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No dropout", "Dropout"],
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# seleccion del modelo ganador por F1-Score
# usamos F1 porque balancea precision y recall, que es lo que nos importa en un problema de deteccion de dropout

winner_name = df_comparison["F1-Score"].idxmax()
print(f"Modelo ganador: {winner_name}")
print(f"  F1-Score : {df_comparison.loc[winner_name, 'F1-Score']}")
print(f"  ROC-AUC  : {df_comparison.loc[winner_name, 'ROC-AUC']}")
print(f"  Accuracy : {df_comparison.loc[winner_name, 'Accuracy']}")

In [ ]:
# curvas ROC

from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
for (name, _, y_prob) in results_opt:
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# guardamos el modelo ganador con pickle

import pickle

winner_models = {
    "Logistic Regression": best_lr,
    "Random Forest": best_rf,
    "Gradient Boosting": best_gb,
}
best_model = winner_models[winner_name]

pickle.dump(best_model, open("../models/best_model.sav", "wb"))

# verificamos que se cargue bien
loaded_model = pickle.load(open("../models/best_model.sav", "rb"))
y_pred_check = loaded_model.predict(X_test_sel)
print(f"Modelo guardado: {winner_name}")
print(f"Accuracy del modelo cargado: {accuracy_score(y_test, y_pred_check):.4f}")

In [ ]:
# guardamos tambien el scaler y el selector para usarlos en la app de streamlit

pickle.dump(scaler, open("../models/scaler.sav", "wb"))
pickle.dump(selector, open("../models/selector.sav", "wb"))
print("Scaler y selector guardados en ../models/")

In [ ]:
# CONCLUSION paso 7: se probaron 3 modelos (logistic regression, random forest y gradient boosting).
# primero sin optimizar para tener una linea base y despues optimizando los hiperparametros con GridSearchCV
# y RandomizedSearchCV. Se eligio el ganador por F1-Score porque en un problema de deteccion de dropout
# necesitamos balancear precision (no alarmar sin razon) y recall (no dejar pasar estudiantes que si van a abandonar).
# el modelo, scaler y selector se guardaron con pickle para usarlos en la app de streamlit

In [ ]:
# el codigo de la app de streamlit esta en app.py
# carga el modelo, scaler y selector con pickle
# el usuario mete los datos del estudiante con sliders
# y la app predice si va a abandonar o no